In [1]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.3 MB/s eta 0:00:00


# Get Dataset

In [2]:
!git clone https://huggingface.co/datasets/Sparkplugx1904/voiceoftrisma-voice-datasets/ temp
!mv temp/* ./
!rm -rf temp

Cloning into 'temp'...
remote: Enumerating objects: 293, done.
remote: Total 293 (delta 0), reused 0 (delta 0), pack-reused 293 (from 1)
Receiving objects: 100% (293/293), 54.34 KiB | 9.06 MiB/s, done.
Filtering content: 100% (283/283), 1.43 GiB | 135.57 MiB/s, done.


In [3]:
!tar -xzf Common_Voice_Scripted_Speech_23.0_-_Indonesian.tar.gz
!ls ./cv-corpus-23.0-2025-09-05/id

clip_durations.tsv  other.tsv	  unvalidated_sentences.tsv
clips		    reported.tsv  validated_sentences.tsv
dev.tsv		    test.tsv	  validated.tsv
invalidated.tsv     train.tsv


In [4]:
!mv ./cv-corpus-23.0-2025-09-05/id/* ./
!rm -r ./cv-corpus-23.0-2025-09-05/

In [5]:
!mv ./VOT-Denpasar_18-12-25-0/clips/* ./clips
!mv ./VOT-Denpasar_18-12-25-0/metadata.tsv ./

# Import Library 

In [6]:
from datasets import load_dataset, Audio, Features, Value, Dataset
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import torch
import pandas as pd
import librosa


2025-12-24 03:34:53.984886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766547294.222146      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766547294.275726      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766547294.888012      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766547294.888059      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766547294.888061      23 computation_placer.cc:177] computation placer alr

# Data loading and processing

In [7]:
import pandas as pd
from datasets import Dataset, Value

# 1. Load Dataframes
train_df = pd.read_csv("train.tsv", sep='\t')
test_df = pd.read_csv("test.tsv", sep='\t')
validated_df = pd.read_csv("validated.tsv", sep='\t')
metadata_df = pd.read_csv("metadata.tsv", sep='\t') 

# 2. Samakan format kolom (Hanya ambil path dan sentence)
train_df = train_df[["path", "sentence"]]
test_df = test_df[["path", "sentence"]]
validated_df = validated_df[["path", "sentence"]]
metadata_df = metadata_df[["path", "sentence"]] 

# 3. Tambahkan prefix "clips/" ke path
train_df["path"] = "clips/" + train_df["path"].astype(str)
test_df["path"] = "clips/" + test_df["path"].astype(str)
validated_df["path"] = "clips/" + validated_df["path"].astype(str)
metadata_df["path"] = "clips/" + metadata_df["path"].astype(str) 

# 4. GABUNGKAN metadata.tsv ke dalam train_df
# Langkah ini menyatukan validated.tsv dan metadata.tsv menjadi satu set latihan
train_df = pd.concat([train_df, metadata_df], ignore_index=True)

# Acak data agar distribusi metadata merata
train_df = train_df.sample(frac=1).reset_index(drop=True)

# 5. Konversi ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
validated_dataset = Dataset.from_pandas(validated_df, preserve_index=False)

# 6. Cast Column path menjadi string
train_dataset = train_dataset.cast_column("path", Value("string"))
test_dataset = test_dataset.cast_column("path", Value("string"))
validated_dataset = validated_dataset.cast_column("path", Value("string"))



Casting the dataset:   0%|          | 0/5255 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3691 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/30218 [00:00<?, ? examples/s]

# Load Whisper Processor and Dataset

In [8]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

# 1. Pastikan Processor dimuat terlebih dahulu
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="indonesian", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="indonesian", task="transcribe")

# 2. Fungsi pemrosesan tunggal
def prepare_dataset(batch):
    # Mengonversi teks (sentence) menjadi token ID (labels)
    # Gunakan truncation=True untuk menghindari error memori pada kalimat yang terlalu panjang
    batch["labels"] = processor.tokenizer(batch["sentence"], truncation=True).input_ids
    return batch

# 3. Jalankan pemetaan (Map)
# PENTING: Kita hanya menghapus "sentence". Kolom "path" HARUS TETAP ADA 
# karena CollatorOnTheFly membutuhkan path untuk memuat audio.
print("Melakukan tokenisasi dataset...")

train_dataset = train_dataset.map(prepare_dataset, remove_columns=["sentence"])
test_dataset = test_dataset.map(prepare_dataset, remove_columns=["sentence"])
validated_dataset = validated_dataset.map(prepare_dataset, remove_columns=["sentence"])

# 4. Verifikasi hasil
print(f"Kolom yang tersedia sekarang: {train_dataset.column_names}")
# Output harusnya: ['path', 'labels']

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Melakukan tokenisasi dataset...


Map:   0%|          | 0/5255 [00:00<?, ? examples/s]

Map:   0%|          | 0/3691 [00:00<?, ? examples/s]

Map:   0%|          | 0/30218 [00:00<?, ? examples/s]

Kolom yang tersedia sekarang: ['path', 'labels']


# Load a Pre-Trained Checkpoint

In [9]:
# Load a Pre-Trained Checkpoint
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [10]:
class CollatorOnTheFly:
    def __init__(self, processor, sampling_rate=16000):
        self.processor = processor
        self.sr = sampling_rate
        self.pad = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id

    def __call__(self, features):
        # load audio on-the-fly
        audios = []
        labels = []

        for f in features:
            try:
                audio, _ = librosa.load(f["path"], sr=self.sr)
            except:
                audio = np.zeros(self.sr, dtype=np.float32)

            audios.append(audio)
            labels.append(torch.tensor(f["labels"], dtype=torch.long))

        # extract features
        inputs = self.processor.feature_extractor(audios, sampling_rate=self.sr)
        feats = [torch.tensor(x) for x in inputs["input_features"]]

        # pad audio features
        feats = torch.nn.utils.rnn.pad_sequence(feats, batch_first=True, padding_value=0.0)

        # pad labels
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=self.pad)

        return {
            "input_features": feats,
            "labels": labels
        }


In [11]:
data_collator = CollatorOnTheFly(processor)

In [12]:
import librosa
import numpy as np
import torch

# 1. Definisikan fungsi pendukung (Tetap dipertahankan jika dibutuhkan nanti)
def load_audio_file(path, sr=16000):
    audio, _ = librosa.load(path, sr=sr)
    return audio

# 2. Cek apakah kolom 'labels' sudah ada untuk menghindari ValueError
# Jika sudah ada, kita tidak perlu melakukan .map() lagi
if "labels" not in train_dataset.column_names:
    def prepare_labels(batch):
        batch["labels"] = processor.tokenizer(batch["sentence"], truncation=True).input_ids
        return batch

    print("Kolom 'labels' belum ada. Melakukan tokenisasi...")
    train_dataset = train_dataset.map(prepare_labels, batched=True, batch_size=128, remove_columns=["sentence"])
    test_dataset = test_dataset.map(prepare_labels, batched=True, batch_size=128, remove_columns=["sentence"])
    validated_dataset = validated_dataset.map(prepare_labels, batched=True, batch_size=128, remove_columns=["sentence"])
else:
    print("Dataset sudah memiliki kolom 'labels'. Melewati proses tokenisasi untuk mencegah error.")

# 3. Verifikasi kolom akhir sebelum masuk ke Training
print(f"Kolom yang tersedia untuk Trainer: {train_dataset.column_names}")
# Hasil yang benar adalah: ['path', 'labels']

Dataset sudah memiliki kolom 'labels'. Melewati proses tokenisasi untuk mencegah error.
Kolom yang tersedia untuk Trainer: ['path', 'labels']


In [13]:
print(train_df.loc[10, "path"])
print(train_df.loc[10, "sentence"])


clips/common_voice_id_26008983.mp3
Serangga telah dilepaskan untuk kontrol biologis gulma buaya.


# Collator

In [14]:
import librosa
import numpy as np
import torch

class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor
        self.sr = 16000
        self.pad = processor.tokenizer.pad_token_id

    def __call__(self, features):
        # 1. Feature Extraction (On-the-fly)
        input_features = [{
            "input_features": self.processor.feature_extractor(
                librosa.load(feature["path"], sr=self.sr)[0],
                sampling_rate=self.sr
            ).input_features[0]
        } for feature in features]

        # 2. Padding audio features
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 3. Padding labels
        labels = [torch.tensor(feature["labels"]) for feature in features]
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=self.pad
        )

        batch["labels"] = labels_padded
        return batch


# Launch

In [15]:
import os
import gc
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback

# --- 1. SUPER AGGRESSIVE CLEANUP ---
torch.cuda.empty_cache()
gc.collect()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- 2. CONFIGURATION ---
USE_CHECKPOINTING = False 
model.config.use_cache = False 

# --- 3. TRAINING ARGUMENTS (OPTIMIZED FOR 15 EPOCHS & 11H LIMIT) ---
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-finetune-id",
    
    # Menaikkan batch fisik ke 4 agar proses it/s jauh lebih kencang
    # Jika OOM (Memory Full), kembalikan per_device_train_batch_size ke 2
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4, 
    
    per_device_eval_batch_size=2,
    
    # --- Optimization (Brutal Mode) ---
    learning_rate=1e-5,              # LR Agresif untuk perubahan akurasi drastis
    warmup_steps=100,                # Pemanasan dipercepat
    num_train_epochs=30,             # Target 15 Epoch sesuai permintaan
    lr_scheduler_type="cosine",      # Penurunan LR halus untuk hasil akhir presisi
    weight_decay=0.01,
    
    # --- Memory Safety ---
    fp16=True,                       
    gradient_checkpointing=USE_CHECKPOINTING, 
    
    # --- Evaluation ---
    predict_with_generate=False,     
    eval_accumulation_steps=1,       
    
    # --- Logging & Strategy (Disesuaikan agar hemat waktu kernel) ---
    eval_strategy="steps",           # Diperlukan untuk load_best_model
    logging_steps=5,                # Log lebih rapat agar progres terpantau
    save_strategy="steps",
    save_steps=1000,                 # Evaluasi & Save tiap 1000 step agar tidak buang waktu
    eval_steps=1000,
    report_to=["tensorboard"],
    
    # Mengambil model dengan Loss terendah untuk akurasi maksimal
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    
    save_total_limit=1,              # Hemat penyimpanan
    remove_unused_columns=False,
    dataloader_num_workers=2,        # Meningkatkan kecepatan loading data
)

# --- 4. CALLBACK PEMBERSIH ---
class ClearMemoryCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()

# --- 5. INITIALIZE TRAINER ---
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    callbacks=[ClearMemoryCallback()]
)

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.gradient_checkpointing = False

print("Starting Training...")
print(f"Learning Rate: {training_args.learning_rate}")

# --- 6. TRAIN ---
trainer.train()

Starting Training...
Learning Rate: 1e-05


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss
1000,0.054700,0.358026
2000,0.033900,0.418516
3000,0.021300,0.440449
4000,0.001000,0.447412
5000,0.000500,0.450131
6000,0.000300,0.466908
7000,0.000300,0.461294
8000,0.000200,0.473013
9000,0.000200,0.476772


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=9870, training_loss=0.05216275744669502, metrics={'train_runtime': 42723.889, 'train_samples_per_second': 3.69, 'train_steps_per_second': 0.231, 'total_flos': 4.5495488360448e+19, 'train_loss': 0.05216275744669502, 'epoch': 30.0})

In [16]:
trainer.save_model("./whisper-finetune-id")

In [17]:
processor.save_pretrained("./whisper-finetune-id")
tokenizer.save_pretrained("./whisper-finetune-id")

('./whisper-finetune-id/tokenizer_config.json',
 './whisper-finetune-id/special_tokens_map.json',
 './whisper-finetune-id/vocab.json',
 './whisper-finetune-id/merges.txt',
 './whisper-finetune-id/normalizer.json',
 './whisper-finetune-id/added_tokens.json')